# GPAT

Gridded Plume Analysis Tool (GPAT) modelling framework. This simulates flight trajectories, estimates fuel burn and emissions, models dispersion effects, and aggregates plume data to a common Eulerian grid for further photochemical and microphysical processing.

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
from dataclasses import asdict
from pycontrails.models.gpat.gpat import GPAT, SimParams, FlParams, PlParams, MetParams, ChemParams, dict_to_dataclass
import os
import holoviews as hv
import hvplot.pandas
import hvplot.xarray

In [2]:
# global simulation parameters
sim_params = {
    "t_fl": (pd.to_datetime("2022-01-20 13:00:00"), pd.Timedelta(minutes=1), pd.Timedelta(hours=1)),# (start time, time step, run time)
    "t_pl": (pd.to_datetime("2022-01-20 13:00:00"), pd.Timedelta(minutes=1), pd.Timedelta(hours=2)),# (start time, time step, max age)
    "t_sim": (pd.to_datetime("2022-01-20 12:00:00"), pd.Timedelta(seconds=20), pd.Timedelta(hours=4)),# (start time, time step, run time)
    "t_out": (pd.to_datetime("2022-01-20 12:00:00"), pd.Timedelta(minutes=1), pd.Timedelta(hours=4)),# (start time, time step, run time)
    "lat_bounds": (0.0, 1.0),  # lat bounds [deg]
    "lon_bounds": (0.0, 1.0),  # lon bounds [deg]
    "alt_bounds": (10000, 11000),  # alt bounds [m]
    "hres_sim_c": 0.05,  # coarse horizontal resolution [deg]
    "vres_sim_c": 500,  # coarse vertical resolution [m]
    "hres_sim_f": 0.001,  # fine horizontal resolution [deg]
    "vres_sim_f": 100,  # fine vertical resolution [m]

    "run_path": "/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/",
    "data_path": "/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/", # "/projects/Impact_of_aviation_on_climate
    "job_id": "GPAT_Feb_2026_test_2_ac",
}

In [3]:
#flight trajectory parameters
fl_params = {
    "mode": "synthetic",
    "file": None,  # flight trajectory file

    "ac_type": "A320",  # aircraft type
    "fl0_speed": 150.0,  # m/s
    "fl0_heading": 45.0,  # deg
    "fl0_coords0": (0.1, 0.1, 10500),  # lat, lon, alt [deg, deg, m]
    "sep_dist": (10000, 5000, 0),  # dx, dy, dz [m]
    "n_ac": 2,  # number of aircraft
}

In [4]:
# plume dispersion parameters
pl_params = {
    "depth": 50.0,  # initial plume depth, [m]
    "width": 50.0,  # initial plume width, [m]
    "verbose_outputs": False,  # print verbose outputs
    "shear": 0.01,  # shear [m/s]
    "n_slices": 3,  # number of slices in the plume
    "f_max": 0.99,  # maximum fraction of total emissions in any slice
    "output_pl_slices": True,  # output plume slices to netCDF
    }

In [5]:
# meteorology parameters
met_params = {
    "eastward_wind": 0.0,  # m/s
    "northward_wind": 0.0,  # m/s
    "lagrangian_tendency_of_air_pressure": 0.0,  # m/s
}

In [6]:
# chemistry parameters
chem_params = {
    "run_chem": True,
    "species_emi": ("NO", "CO", "SO2"),
    # "species_pl": ("NO", "CO", "SO2"),
    "species_pl": ("NO", "NO2", "O3", "NO3", "N2O5",
                      "HNO3", "HONO", "HO2NO2","PAN", 
                      "CH3O2NO2","H2O2", "CH3OOH",
                      "CO", "CH4", "HCHO", "SO2", "SA"),
    "species_out": ("O3", "NO2", "NO", "NO3", "N2O5", 
                    "HNO3", "HONO", "HO2", "OH", "H2O2",
                    "CO", "CH4", "CH3O2","HO2NO2", "PAN", "SO2" )
}

In [7]:
sim_params = SimParams(**sim_params)
fl_params = FlParams(**fl_params)
pl_params = PlParams(**pl_params)
met_params = MetParams(**met_params)
chem_params = ChemParams(**chem_params)

gpat = GPAT(sim_params, fl_params, pl_params, met_params, chem_params)

In [8]:
gpat.preprocess_gpat()

/home/ktait98/miniconda3/envs/contrails/lib/python3.12/site-packages/xarray/core/duck_array_ops.py:234: UserWarning: no explicit representation of timezones available for np.datetime64
  return data.astype(dtype, **kwargs)


flight 0 done
flight 1 done


/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/gpat.py:627: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  fl[i].dataframe[column] = fl[i].dataframe[column].fillna(method="ffill")
/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/gpat.py:690: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  fl[i][column] = fl[i][column].fillna(method="ffill")
/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/gpat.py:690: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  fl[i][column] = fl[i][column].fillna(method="ffill")
/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/gpat.py:903: RuntimeWarning: invalid value encountered in cast
  age_seconds = np.where(np.isnat(age_values), 0, (age_values / np

Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/inputs/GPAT_Feb_2026_test_2_ac/boxm_ds.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/inputs/GPAT_Feb_2026_test_2_ac/fl_ds.nc
active seg flags for seg 1 and all ts: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/inputs/GPAT_Feb_2026_test_2_ac/pl_ds.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/outputs/GPAT_Feb_2026_test_2_ac/boxm_out.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/outputs/GPAT_Feb_2026_test_2_ac/patch_table.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/outputs/GPAT_Feb_2026_test_2_ac/pl_

In [9]:
gpat.eval()

 FL_DS SUMMARY:
   NSEG =           30
   IS_OPEN =  T
   NCID =        65536
 ACTIVE_SEG_FLAG:            1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1           1    

In [10]:
fl_ds = xr.open_dataset(f"{gpat.inputs_job}/fl_ds.nc")
fl_ds.longitude.values

array([0.1       , 0.15717143, 0.21434285, 0.27151428, 0.32868571,
       0.38585713, 0.44302856, 0.50019999, 0.55737141, 0.61454284,
       0.67171427, 0.72889915, 0.78608403, 0.8432689 , 0.90045378,
       0.95763866, 0.19528105, 0.25245248, 0.3096239 , 0.36679533,
       0.42396676, 0.48113819, 0.53830961, 0.59548104, 0.65265247,
       0.70982389, 0.76699532, 0.8241802 , 0.88136508, 0.93854996])

In [19]:
from IPython.display import clear_output
clear_output(wait=True)

pl_ds = xr.open_dataset(f"{gpat.inputs_job}/pl_ds.nc")
pl_ds.sel(ht="tail").longitude.values

array([[0.1       , 0.1       , 0.1       , ...,        nan,        nan,
               nan],
       [       nan, 0.15717143, 0.15717143, ...,        nan,        nan,
               nan],
       [       nan,        nan, 0.21434285, ...,        nan,        nan,
               nan],
       ...,
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [       nan,        nan,        nan, ..., 0.93854996,        nan,
               nan]], shape=(30, 135))

In [12]:
pl_ds.active_seg_flag.isel(seg_id=0).values


array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0])

In [13]:
for t in pl_ds.time.values:
    active_mask = pl_ds["active_seg_flag"][:, pl_ds.get_index("time").get_loc(t)]
    active_seg_ids = pl_ds["seg_id"].values[active_mask.values]
    

# pl_ds.active_seg_flag.isel(seg_id=0).values

In [14]:
boxm_ds = xr.open_dataset(f"{gpat.inputs_job}/boxm_ds.nc")
boxm_ds.longitude_c.values

array([0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025,
       0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025,
       0.025, 0.025, 0.075, 0.075, 0.075, 0.075, 0.075, 0.075, 0.075,
       0.075, 0.075, 0.075, 0.075, 0.075, 0.075, 0.075, 0.075, 0.075,
       0.075, 0.075, 0.075, 0.075, 0.125, 0.125, 0.125, 0.125, 0.125,
       0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125,
       0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.175, 0.175, 0.175,
       0.175, 0.175, 0.175, 0.175, 0.175, 0.175, 0.175, 0.175, 0.175,
       0.175, 0.175, 0.175, 0.175, 0.175, 0.175, 0.175, 0.175, 0.225,
       0.225, 0.225, 0.225, 0.225, 0.225, 0.225, 0.225, 0.225, 0.225,
       0.225, 0.225, 0.225, 0.225, 0.225, 0.225, 0.225, 0.225, 0.225,
       0.225, 0.275, 0.275, 0.275, 0.275, 0.275, 0.275, 0.275, 0.275,
       0.275, 0.275, 0.275, 0.275, 0.275, 0.275, 0.275, 0.275, 0.275,
       0.275, 0.275, 0.275, 0.325, 0.325, 0.325, 0.325, 0.325, 0.325,
       0.325, 0.325,

In [15]:
# Matplotlib time slider (ipympl): trajectories + plume width (tail only)
%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import ipywidgets as widgets
from IPython.display import display


# gpat.analysis.plot_plumes()

In [16]:
gpat.analysis.load_output_datasets()
# gpat.pl_out = xr.open_dataset(f"{gpat.outputs_job}/pl_out.nc")
gpat.pl_ds


<xarray.Dataset> Size: 1MB
Dimensions:          (seg_id: 30, time: 135, ht: 2, species_emi: 3)
Coordinates:
    flight_id        (seg_id) int64 240B 1 1 1 1 1 1 1 1 1 ... 2 2 2 2 2 2 2 2 2
    waypoint         (seg_id) int64 240B 1 2 3 4 5 6 7 8 ... 8 9 10 11 12 13 14
  * seg_id           (seg_id) int64 240B 1 2 3 4 5 6 7 ... 24 25 26 27 28 29 30
  * time             (time) object 1kB '2022-01-20T13:00:00Z' ... '2022-01-20...
    species_emi_num  (species_emi) int64 24B 8 11 16
  * species_emi      (species_emi) object 24B 'NO' 'CO' 'SO2'
    active_seg_flag  (seg_id, time) int64 32kB 1 1 1 1 1 1 1 1 ... 1 1 1 1 1 0 0
    time_rel_s       (time) int64 1kB 3600 3660 3720 3780 ... 11520 11580 11640
    time_idx         (time) int64 1kB 181 184 187 190 193 ... 574 577 580 583
  * ht               (ht) <U4 32B 'tail' 'head'
Data variables: (12/13)
    age              (seg_id, time) <U21 340kB '60000000000 nanosecon' ... 'NaT'
    longitude        (seg_id, ht, time) float64 65kB 0.1 0.1 0.1 ... nan nan
    latitude         (seg_id, ht, time) float64 65kB 0.1 0.1 0.1 ... nan nan
    level            (seg_id, ht, time) float64 65kB 244.7 244.7 ... nan nan
    width            (seg_id, ht, time) float64 65kB 117.6 162.2 ... nan nan
    depth            (seg_id, ht, time) float64 65kB 109.6 118.3 ... nan nan
    ...               ...
    sigma_yy         (seg_id, ht, time) float64 65kB 1.73e+03 3.29e+03 ... nan
    sigma_yz         (seg_id, ht, time) float64 65kB 825.1 1.8e+03 ... nan nan
    sigma_zz         (seg_id, ht, time) float64 65kB 1.5e+03 1.75e+03 ... nan
    altitude         (seg_id, ht, time) float64 65kB 1.05e+04 1.05e+04 ... nan
    emi_pl_mass      (seg_id, species_emi) float64 720B 0.3131 ... 0.0292
    age_s            (seg_id, time) int64 32kB 60 120 180 240 ... 7140 7200 0 0
Attributes: (12/13)
    nseg:              30
    ts_fl:             60.0
    ts_pl:             60.0
    ts_sim:            20.0
    ts_out:            60.0
    species_emi:       ('NO', 'CO', 'SO2')
    ...                ...
    species_emi_num:   [ 8 11 16]
    species_pl_num:    [  8   4   6   5   7  14  13  15 198 217  12 144  11  ...
    n_slices:          3
    f_max:             0.99
    output_pl_slices:  1
    description:       Emission species mass in plume segments

In [17]:
gpat.pl_out


<xarray.Dataset> Size: 6MB
Dimensions:         (seg_id: 30, time: 241, species_pl: 17, ht: 2, slice_id: 3,
                     corner_id: 4, coord: 3)
Coordinates:
    flight_id       (seg_id) int64 240B ...
    waypoint        (seg_id) int64 240B ...
  * seg_id          (seg_id) int64 240B 1 2 3 4 5 6 7 8 ... 24 25 26 27 28 29 30
  * time            (time) <U20 19kB '2022-01-20T12:00:00Z' ... '2022-01-20T1...
  * species_pl      (species_pl) <U8 544B 'NO' 'NO2' 'O3' ... 'HCHO' 'SO2' 'SA'
  * ht              (ht) <U4 32B 'tail' 'head'
  * slice_id        (slice_id) int64 24B 1 2 3
  * corner_id       (corner_id) <U2 32B 'BL' 'TL' 'TR' 'BR'
  * coord           (coord) <U5 60B 'lon_m' 'lat_m' 'alt_m'
    species_pl_num  (species_pl) int64 136B ...
    time_rel_s      (time) int64 2kB ...
    time_idx        (time) int64 2kB ...
Data variables:
    y_half          (seg_id, ht, slice_id, time) float64 347kB ...
    z_half          (seg_id, ht, slice_id, time) float64 347kB ...
    m_frac          (slice_id) float64 24B ...
    w_slice         (slice_id) float64 24B ...
    slice_poly      (seg_id, ht, slice_id, corner_id, coord, time) float64 4MB ...
    pl_mass         (seg_id, species_pl, time) float64 983kB ...
Attributes:
    description:  Plume segment output for BOXM

In [18]:
# print(gpat.pl_out.slice_poly.isel(time=62, slice_id=0, corner_id=0).values)# gpat.pl_out.sel(slice_id=1, ht="tail", time='2022-01-20T13:40:00Z')["y_half"].values
gpat.analysis.plot_plumes_3d()

NameError: name 'times' is not defined

In [ ]:
from IPython.display import clear_output
clear_output(wait=True)

boxm_out = xr.open_dataset(f"{gpat.outputs_job}/boxm_out.nc")

boxm_out

In [ ]:
gpat.patch_table

In [ ]:
from IPython.display import clear_output
clear_output(wait=True)

pl_out = xr.open_dataset(f"{gpat.outputs_job}/pl_out.nc")
pl_out

In [ ]:
pl_out.slice_poly.isel(time=100, seg_id=0, slice_id=0, ht=0).values

In [ ]:
from IPython.display import clear_output
clear_output(wait=True)

np.set_printoptions(threshold=np.inf, linewidth=2000)
print(pl_out["pl_mass"].isel(species_pl=15, time=slice(60, 80)).values)

In [ ]:
pl_ds["species_emi_num"].values
pl_ds["emi_pl_mass"].isel(seg_id=0).to_pandas()
pl_ds["emi_pl_mass"].max("seg_id").to_pandas()

print(pl_ds["emi_pl_mass"].max("seg_id").to_pandas().to_string())

In [ ]:
pl_out["pl_mass"].max(("seg_id","time")).to_pandas()

In [ ]:
pd.DataFrame({
    "species_emi": pl_ds["species_emi"].values,
    "species_emi_num": pl_ds["species_emi_num"].values,
})

pd.DataFrame({
    "species_pl": pl_out["species_pl"].values,
    "species_pl_num": pl_out["species_pl_num"].values,
})

print(pd.DataFrame({
    "species_emi": pl_ds["species_emi"].values,
    "species_emi_num": pl_ds["species_emi_num"].values,
}).to_string(index=False))

print(pd.DataFrame({
    "species_pl": pl_out["species_pl"].values,
    "species_pl_num": pl_out["species_pl_num"].values,
}).to_string(index=False))